## Sentiment and Thematic Analysis
1. Sentiment Analysis
Library Options: Start with distilbert-base-uncased-finetuned-sst-2-english, or use simpler libraries like VADER or TextBlob for initial analysis.

In [3]:
import pandas as pd
from textblob import TextBlob

In [4]:
# Load your cleaned review data
df = pd.read_csv('../data/processed_bank_reviews.csv')

In [5]:
def analyze_sentiment(text):
    if pd.isna(text):
        return 'neutral', 0.0  # Default neutral score
    blob = TextBlob(text)
    return 'positive' if blob.sentiment.polarity > 0 else 'negative' if blob.sentiment.polarity < 0 else 'neutral', blob.sentiment.polarity

In [6]:
df[['sentiment_label', 'sentiment_score']] = df['review'].apply(analyze_sentiment).apply(pd.Series)

In [7]:
df.to_csv('../data/sentiment_analysis_results.csv', index=False)

2. Thematic Analysis

Thematic analysis is a qualitative research method used to identify, analyze, and report patterns (themes) within data. In the context of user reviews, thematic analysis helps to understand common sentiments and concerns expressed by users, allowing you to categorize feedback effectively.

## Steps for Thematic Analysis
Data Preparation: Ensure your data is clean and structured.
Keyword Extraction: Identify keywords or phrases that frequently appear in the reviews.
Theme Identification: Group related keywords into broader themes.
Documentation: Summarize the themes and provide examples from the reviews.


In [8]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

In [22]:
sentiment_df = pd.read_csv('../data/sentiment_analysis_results.csv')
print("Columns in sentiment_df:", sentiment_df.columns)


Columns in sentiment_df: Index(['review', 'rating', 'date', 'bank', 'source', 'sentiment_label',
       'sentiment_score'],
      dtype='object')


In [23]:
sentiment_df['review_id'] = sentiment_df.index + 1  # Create a simple ID based on index

nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    doc = nlp(text.lower())
    return ' '.join([token.lemma_ for token in doc if not token.is_stop and token.is_alpha])

# Apply preprocessing
sentiment_df['cleaned_reviews'] = sentiment_df['review'].apply(preprocess_text)


In [24]:

# TF-IDF extraction
tfidf = TfidfVectorizer(max_features=100)
tfidf_matrix = tfidf.fit_transform(sentiment_df['cleaned_reviews'])


In [28]:
# KMeans clustering
num_clusters = 5
kmeans = KMeans(n_clusters=num_clusters, random_state=42)
kmeans.fit(tfidf_matrix)

# Check if clustering was successful
if hasattr(kmeans, 'labels_'):
    sentiment_df['identified_theme'] = kmeans.labels_
else:
    print("Clustering failed to produce labels.")

# Debugging: Check the unique labels generated
print("Unique labels from clustering:", set(kmeans.labels_))


Unique labels from clustering: {np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4)}


In [29]:
theme_mapping = {
    0: 'Account Access Issues',
    1: 'Transaction Performance',
    2: 'User Interface & Experience',
    3: 'Customer Support',
    4: 'Feature Requests'
}

# Map the cluster labels to themes
sentiment_df['identified_theme'] = sentiment_df['identified_theme'].map(theme_mapping)

# Step 3: Save combined results
print("Columns before saving:", sentiment_df.columns)

Columns before saving: Index(['review', 'rating', 'date', 'bank', 'source', 'sentiment_label',
       'sentiment_score', 'review_id', 'cleaned_reviews', 'identified_theme'],
      dtype='object')


In [30]:
# Save combined results
sentiment_df[['review_id', 'review', 'sentiment_label', 'sentiment_score', 'identified_theme']].to_csv('../data/combined_analysis_results.csv', index=False)